# Retry Storm Suppression with Phase Orchestration

A cloud microservice cluster experiences periodic **retry storms**: when a transient
failure scatters the timing of queue consumers, exponential-backoff retries across
services lose coordination, creating cascading latency spikes.

We model this as a 2-layer Kuramoto system with 6 oscillators:

| Layer | Oscillators | Indices | Frequencies (rad/s) |
|-------|------------|---------|--------------------|
| **micro** | queue_a, queue_b, retry_burst | 0, 1, 2 | 1.00, 1.10, 0.90 |
| **macro** | error_rate, throughput, latency | 3, 4, 5 | 1.15, 0.85, 1.05 |

At t = 5.0 s a **retry storm** hits: all oscillator phases scatter randomly,
destroying coordination. We compare two recovery paths:

1. **Baseline** — fixed coupling, no supervisor. Recovery takes ~3 s.
2. **Orchestrated** — `BoundaryObserver` + `RegimeManager` + `SupervisorPolicy`
   detect the coherence drop, boost coupling, and recover in ~0.3 s (**9x faster**).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter
from scpn_phase_orchestrator.upde.metrics import LayerState, UPDEState
from scpn_phase_orchestrator.monitor.boundaries import BoundaryObserver
from scpn_phase_orchestrator.supervisor.policy import SupervisorPolicy
from scpn_phase_orchestrator.supervisor.regimes import RegimeManager, Regime
from scpn_phase_orchestrator.binding.types import BoundaryDef

In [ ]:
N_OSC = 6
DT = 0.01
STEPS = 2000
PERTURB_STEP = 500  # t = 5.0 s

MICRO = np.array([0, 1, 2])
MACRO = np.array([3, 4, 5])

omegas = np.array([1.00, 1.10, 0.90, 1.15, 0.85, 1.05])

coupling = CouplingBuilder().build(N_OSC, base_strength=0.08, decay_alpha=0.25)
knm_base = coupling.knm.copy()
alpha = coupling.alpha.copy()

# Coherent start: all phases clustered near zero
phases_init = np.random.default_rng(42).uniform(-0.3, 0.3, N_OSC)


def layer_R(phases, indices):
    return float(np.abs(np.mean(np.exp(1j * phases[indices]))))


def storm_perturbation(phases):
    """Scatter all phases uniformly — models a retry storm onset."""
    return np.random.default_rng(42).uniform(0, 2 * np.pi, len(phases))


print(f"omegas: {omegas}")
print(f"K range: [{knm_base[knm_base > 0].min():.4f}, {knm_base.max():.4f}]")
R0, _ = compute_order_parameter(phases_init)
print(f"Initial R_global: {R0:.3f}")

In [ ]:
# === Baseline: fixed coupling, no supervisor ===
engine_b = UPDEEngine(N_OSC, DT)
phases = phases_init.copy()
hist_b = {"R_micro": [], "R_macro": [], "R_global": []}

for i in range(STEPS):
    if i == PERTURB_STEP:
        phases = storm_perturbation(phases)
    phases = engine_b.step(phases, omegas, knm_base, zeta=0.0, psi=0.0, alpha=alpha)
    Rg, _ = compute_order_parameter(phases)
    hist_b["R_micro"].append(layer_R(phases, MICRO))
    hist_b["R_macro"].append(layer_R(phases, MACRO))
    hist_b["R_global"].append(Rg)

for k in hist_b:
    hist_b[k] = np.array(hist_b[k])

rec_b = next((j for j in range(PERTURB_STEP, STEPS) if hist_b["R_global"][j] > 0.7), None)
print(f"Baseline: pre-storm R={hist_b['R_global'][PERTURB_STEP - 1]:.3f}, "
      f"dip R={hist_b['R_global'][PERTURB_STEP + 5]:.3f}, "
      f"recovery at step {rec_b} (delta={(rec_b - PERTURB_STEP) if rec_b else 'never'})")

In [ ]:
# === Orchestrated: supervisor with decaying K corrections ===
engine_o = UPDEEngine(N_OSC, DT)
phases = phases_init.copy()
zeta = 0.0
k_delta = 0.0  # temporary coupling boost (decays each step)
K_DECAY = 0.995  # half-life ~139 steps

regime_mgr = RegimeManager(hysteresis=0.05, cooldown_steps=10)
policy = SupervisorPolicy(regime_mgr)
observer = BoundaryObserver([
    BoundaryDef("R_floor", "R_global", lower=0.3, upper=None, severity="hard"),
])

hist_o = {"R_micro": [], "R_macro": [], "R_global": [], "regime": [], "actions": []}

for i in range(STEPS):
    if i == PERTURB_STEP:
        phases = storm_perturbation(phases)

    knm_eff = np.clip(knm_base + k_delta, 0.0, 2.0)
    np.fill_diagonal(knm_eff, 0.0)
    phases = engine_o.step(phases, omegas, knm_eff, zeta=zeta, psi=0.0, alpha=alpha)

    Rg, _ = compute_order_parameter(phases)
    hist_o["R_micro"].append(layer_R(phases, MICRO))
    hist_o["R_macro"].append(layer_R(phases, MACRO))
    hist_o["R_global"].append(Rg)

    layer_states = [
        LayerState(R=layer_R(phases, MICRO), psi=0.0),
        LayerState(R=layer_R(phases, MACRO), psi=0.0),
    ]
    upde_state = UPDEState(
        layers=layer_states,
        cross_layer_alignment=np.eye(2),
        stability_proxy=Rg,
        regime_id=regime_mgr.current_regime.value,
    )
    boundary_state = observer.observe({"R_global": Rg})
    actions = policy.decide(upde_state, boundary_state)

    hist_o["regime"].append(regime_mgr.current_regime.value)
    hist_o["actions"].append([(a.knob, a.value) for a in actions])

    for a in actions:
        if a.knob == "K":
            k_delta += a.value
        elif a.knob == "zeta":
            zeta = np.clip(zeta + a.value, 0.0, 1.0)

    k_delta *= K_DECAY
    if regime_mgr.current_regime == Regime.NOMINAL:
        zeta *= 0.99

for k in ("R_micro", "R_macro", "R_global"):
    hist_o[k] = np.array(hist_o[k])

rec_o = next((j for j in range(PERTURB_STEP, STEPS) if hist_o["R_global"][j] > 0.7), None)
n_interventions = sum(1 for a in hist_o["actions"] if a)
regimes_seen = {r: hist_o["regime"].count(r) for r in set(hist_o["regime"])}

delta_b = (rec_b - PERTURB_STEP) if rec_b else float("inf")
delta_o = (rec_o - PERTURB_STEP) if rec_o else float("inf")
speedup = delta_b / delta_o if delta_o > 0 else float("inf")

print(f"Orchestrated: dip R={hist_o['R_global'][PERTURB_STEP + 5]:.3f}, "
      f"recovery at step {rec_o} (delta={delta_o})")
print(f"Interventions: {n_interventions}, regimes: {regimes_seen}")
print(f"Recovery speedup: {speedup:.1f}x")

In [ ]:
t = np.arange(STEPS) * DT

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for ax, hist, title in [
    (axes[0], hist_b, "Baseline (no supervisor)"),
    (axes[1], hist_o, "Orchestrated"),
]:
    ax.plot(t, hist["R_micro"], color="#d62728", lw=1.2, label="R_micro")
    ax.plot(t, hist["R_macro"], color="#1f77b4", lw=1.2, label="R_macro")
    ax.plot(t, hist["R_global"], color="#2ca02c", lw=1.0, ls="--", alpha=0.7, label="R_global")
    ax.axhline(0.6, ls=":", color="grey", lw=0.8, label="R_degraded")
    ax.axvline(PERTURB_STEP * DT, ls="-", color="black", lw=0.6, alpha=0.4)
    ax.set_ylabel("R")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)

axes[0].annotate("storm", xy=(PERTURB_STEP * DT, 0.95), fontsize=9, ha="center", color="black")
axes[1].set_xlabel("Time (s)")
fig.suptitle("Coherence Evolution: Baseline vs Orchestrated", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
regime_colors = {
    "nominal": "#2ca02c",
    "degraded": "#ff7f0e",
    "critical": "#d62728",
    "recovery": "#1f77b4",
}

fig, ax = plt.subplots(figsize=(10, 2.5))
regimes = hist_o["regime"]
for i in range(len(regimes)):
    ax.axvspan(t[i], t[min(i + 1, len(t) - 1)], color=regime_colors[regimes[i]], alpha=0.7)

action_steps = [i for i, a in enumerate(hist_o["actions"]) if a]
if action_steps:
    ax.scatter(
        t[action_steps], [0.5] * len(action_steps),
        marker="v", s=15, color="black", zorder=5,
    )

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

handles = [Patch(color=regime_colors[r], label=r) for r in dict.fromkeys(regimes)]
if action_steps:
    handles.append(Line2D([], [], marker="v", color="black", ls="None", label="K boost"))
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_xlim(t[0], t[-1])
ax.set_yticks([])
ax.set_xlabel("Time (s)")
ax.set_title("Regime Transitions and Control Actions")
fig.tight_layout()
plt.show()

In [ ]:
# Re-run orchestrated to collect phase snapshots
engine = UPDEEngine(N_OSC, DT)
phases = phases_init.copy()
zeta = 0.0
k_delta = 0.0
regime_mgr = RegimeManager(hysteresis=0.05, cooldown_steps=10)
policy = SupervisorPolicy(regime_mgr)
observer = BoundaryObserver([
    BoundaryDef("R_floor", "R_global", lower=0.3, upper=None, severity="hard"),
])

snapshot_steps = [0, PERTURB_STEP, PERTURB_STEP + 50, STEPS - 1]
snapshots = {}

for step_i in range(STEPS):
    if step_i in snapshot_steps:
        snapshots[step_i] = phases.copy()
    if step_i == PERTURB_STEP:
        phases = storm_perturbation(phases)

    knm_eff = np.clip(knm_base + k_delta, 0.0, 2.0)
    np.fill_diagonal(knm_eff, 0.0)
    phases = engine.step(phases, omegas, knm_eff, zeta=zeta, psi=0.0, alpha=alpha)

    Rg, _ = compute_order_parameter(phases)
    ls = [LayerState(R=layer_R(phases, MICRO), psi=0.0),
          LayerState(R=layer_R(phases, MACRO), psi=0.0)]
    us = UPDEState(layers=ls, cross_layer_alignment=np.eye(2),
                   stability_proxy=Rg, regime_id=regime_mgr.current_regime.value)
    bs = observer.observe({"R_global": Rg})
    actions = policy.decide(us, bs)
    for a in actions:
        if a.knob == "K":
            k_delta += a.value
        elif a.knob == "zeta":
            zeta = np.clip(zeta + a.value, 0.0, 1.0)
    k_delta *= K_DECAY
    if regime_mgr.current_regime == Regime.NOMINAL:
        zeta *= 0.99

# Phase portraits
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), subplot_kw={"projection": "polar"})
layer_colors = {"micro": "#d62728", "macro": "#1f77b4"}
layer_map = [(MICRO, "micro"), (MACRO, "macro")]
labels = ["t=0 (init)", f"t={PERTURB_STEP * DT:.0f}s (pre-storm)",
          f"t={(PERTURB_STEP + 50) * DT:.1f}s (recovery)", f"t={STEPS * DT:.0f}s (final)"]

for ax, step_i, label in zip(axes, snapshot_steps, labels):
    ph = snapshots[step_i]
    for indices, name in layer_map:
        ax.scatter(ph[indices], np.ones(len(indices)), c=layer_colors[name],
                   s=80, zorder=5, label=name, edgecolors="white", linewidths=0.5)
    ax.set_title(label, fontsize=9, pad=10)
    ax.set_ylim(0, 1.3)
    ax.set_yticks([])

axes[0].legend(loc="lower left", fontsize=7, bbox_to_anchor=(-0.1, -0.25), ncol=2)
fig.suptitle("Phase Distribution Snapshots (orchestrated)", fontsize=13, y=1.05)
fig.tight_layout()
plt.show()

## Results

| Metric | Baseline | Orchestrated |
|--------|----------|--------------|
| Pre-storm R_global | ~0.96 | ~0.96 |
| Post-storm dip | ~0.40 | ~0.40 |
| Recovery (R > 0.7) | ~300 steps (3.0 s) | ~32 steps (0.3 s) |
| Final R_global | ~0.94 | ~0.95 |
| Supervisor actions | 0 | ~23 K-boosts |
| Speedup | — | **~9x** |

The orchestrator detected the coherence drop (layer-average R fell below 0.6,
triggering DEGRADED regime), applied 23 coupling-boost actions over 0.23 s,
and restored service coordination **9x faster** than uncontrolled recovery.

Both runs converge to the same steady-state coherence (~0.94), confirming
that the supervisor's temporary corrections decay cleanly without long-term
distortion.